This file has been used to filter berkeley and create "df_filtered.txt"

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("https://waf.cs.illinois.edu/discovery/berkeley.csv")
df = df.sort_values("Major")

# Remove other department
df = df[df["Major"] != "Other"]

gum_df = pd.DataFrame(df)

# df = df[df["Major"] != "Other"]

df["Admission"] = pd.Categorical(df["Admission"], categories=df["Admission"].unique())
df["Major"] = pd.Categorical(df["Major"], categories=df["Major"].unique())
df["Sex"] = pd.Categorical(df["Sex"], categories=df["Sex"].unique())


df = df.drop(columns=["Year"])

gum_df["Y"] = df["Admission"].cat.codes
gum_df["W"] = df["Major"].cat.codes
gum_df["X"] = df["Sex"].cat.codes
gum_df = gum_df[["X", "W", "Y"]]

cat_map = {
    "X": dict(enumerate(df["Sex"].cat.categories)),
    "W": dict(enumerate(df["Major"].cat.categories)),
    "Y": dict(enumerate(df["Admission"].cat.categories)),
}

df_filtered = df[df["Major"].isin(["A", "C"])]
df_filtered.to_csv("df_filtered.csv", index=False)


Sankey Berkeley 

In [4]:
from libraries import *
from functions import *

import sankey
from importlib import reload
reload(sankey)
from sankey import *

In [5]:
berk_dataset = pd.read_csv("https://waf.cs.illinois.edu/discovery/berkeley.csv") 
berk_dataset.head()

,Year,Major,Sex,Admission
0,1973,C,F,Rejected
1,1973,B,M,Accepted
2,1973,Other,F,Accepted
3,1973,Other,M,Accepted
4,1973,Other,M,Rejected


In [14]:
berk2 = berk_dataset.rename(columns={
    "Sex": "X",
    "Admission": "Y",
    "Major": "W",
    "Year": "Z"
})

berk2= pd.read_csv("df_filtered.csv")
berk2 = berk_dataset.rename(columns={
    "Sex": "X",
    "Admission": "Y",
    "Major": "W",
    "Year": "Z"
})


effects=compute_effects_multi(
    berk2,
    x0="M",
    x1="F",
    y="Accepted",
    W_values=berk2["W"].unique(),
    w_cols=["W"],
   # z_cols=["Z"]
)

effects


{'tv': 0.0,
 'tv1': 0.0,
 'te': 0.0,
 'te_linear': 0.008697466350519944,
 'se': 0.0,
 'se_x1': 0.0,
 'se_x0': 0.0,
 'de': 0.008697466350519944,
 'ie': 0.0,
 'ie_f': 0.0,
 'ie_decomp': {},
 'se_decomp': {},
 'ie_decomp_interval': {},
 'se_decomp_interval': {}}

In [7]:
plot_effect_sankey_percent(
    te=effects["te_linear"],          #or effects["te"]
    se=effects["se"],
    ie=effects["ie"], #qui ie o ie_f indifferente! Quando scompongo-> usare solo ie 
    de=effects["de"],
    se_decomp=effects["se_decomp"],
    ie_decomp=effects["ie_decomp"],
    title="Causal decomposition (percent of |TV|)",
)
